# RDP Shortpath Setup – Analyse

Dieses Notebook analysiert den **dedizierten RDP-Shortpath-Session-Host** (`infra/shortpath/`) und prüft, ob der Public-Networks-Pfad (STUN/TURN) korrekt vorbereitet ist.

**Design-Entscheidung:** Public Networks (STUN/TURN) – keine In-Guest-Konfiguration nötig (UDP/TCP sind in Windows Default). Der Managed-Networks-Pfad (UDP 3390 Listener) ist bewusst **nicht** aktiviert (siehe `infra/shortpath/MANAGED-NETWORKS.md`).

**Was geprüft wird:**
1. Variablen + Login-Kontext
2. Existenz der Shortpath-Ressourcen (Host Pool, App Group, Workspace, VM, NIC)
3. Session-Host-Registrierung im Host Pool
4. RBAC der AVD-Gruppe auf den Shortpath-Ressourcen
5. Firewall-Relay-Regel (UDP 3478 → 51.5.0.0/16)
6. Effektive Routen + ausgehende IP der Shortpath-NIC
7. In-Guest: RDP-Transport-Default (UDP) + Managed-Listener-Status
8. Post-Connection-Verifikation (Event ID 135) – optional

> Kernel: **Bash** (Zellen-Sprache `shellscript`). Ausgaben werden nach `output/shortpath-*.json` gespeichert.


## 1. Variablen + Login-Kontext


In [11]:
export PREFIX=cptdazavdvwan
export RG=rg-${PREFIX}
export SUB=$(az account show --query id -o tsv)

# Dedizierter Shortpath-Stack (infra/shortpath/)
export HP=hp-shortpath-${PREFIX}
export DAG=dag-shortpath-${PREFIX}
export WS=ws-shortpath-${PREFIX}
export VM=vm-shortpath-${PREFIX}
export NIC=nic-shortpath-${PREFIX}

# Standard-Stack zum Vergleich (infra/main.bicep)
export HP_BASE=hp-${PREFIX}
export VM_BASE=vm-avd-${PREFIX}
export NIC_BASE=nic-avd-${PREFIX}

mkdir -p output
echo "RG=$RG"
echo "SUB=$SUB"
echo "Shortpath : HP=$HP  VM=$VM  NIC=$NIC"
echo "Standard  : HP=$HP_BASE  VM=$VM_BASE  NIC=$NIC_BASE"


RG=rg-cptdazavdvwan
SUB=ff0bb075-6c44-44ee-bb64-d46ce828c62f
Shortpath : HP=hp-shortpath-cptdazavdvwan  VM=vm-shortpath-cptdazavdvwan  NIC=nic-shortpath-cptdazavdvwan
Standard  : HP=hp-cptdazavdvwan  VM=vm-avd-cptdazavdvwan  NIC=nic-avd-cptdazavdvwan


## 2. Shortpath-Ressourcen prüfen

**Erwartung:** Host Pool, App Group, Workspace, VM und NIC existieren und sind `Succeeded`.


In [3]:
echo "=== Deployment 'shortpath' outputs ==="
az deployment group show -g $RG -n shortpath --query properties.outputs -o json | tee output/shortpath-outputs.json

echo ""
echo "=== Ressourcen-Inventar ==="
{
  az desktopvirtualization hostpool show -g $RG -n $HP --query "{type:type,name:name,state:provisioningState,poolType:hostPoolType,lb:loadBalancerType}" -o json
  az desktopvirtualization applicationgroup show -g $RG -n $DAG --query "{name:name,type:applicationGroupType}" -o json
  az desktopvirtualization workspace show -g $RG -n $WS --query "{name:name,appGroups:applicationGroupReferences}" -o json
  az vm show -g $RG -n $VM --query "{name:name,size:hardwareProfile.vmSize,computer:osProfile.computerName,state:provisioningState}" -o json
  az network nic show -g $RG -n $NIC --query "{name:name,privateIp:ipConfigurations[0].privateIPAddress,subnet:ipConfigurations[0].subnet.id}" -o json
} | tee output/shortpath-inventory.json


=== Deployment 'shortpath' outputs ===


{
  "appGroupName": {
    "type": "String",
    "value": "dag-shortpath-cptdazavdvwan"
  },
  "hostPoolName": {
    "type": "String",
    "value": "hp-shortpath-cptdazavdvwan"
  },
  "vmName": {
    "type": "String",
    "value": "vm-shortpath-cptdazavdvwan"
  },
  "workspaceName": {
    "type": "String",
    "value": "ws-shortpath-cptdazavdvwan"
  }
}

=== Ressourcen-Inventar ===
{
  "lb": "BreadthFirst",
  "name": "hp-shortpath-cptdazavdvwan",
  "poolType": "Pooled",
  "state": null,
  "type": "Microsoft.DesktopVirtualization/hostpools"
}
{
  "name": "dag-shortpath-cptdazavdvwan",
  "type": "Desktop"
}
{
  "appGroups": [
    "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourcegroups/rg-cptdazavdvwan/providers/Microsoft.DesktopVirtualization/applicationgroups/dag-shortpath-cptdazavdvwan"
  ],
  "name": "ws-shortpath-cptdazavdvwan"
}
{
  "computer": "vmsp01",
  "name": "vm-shortpath-cptdazavdvwan",
  "size": "Standard_D2s_v5",
  "state": "Succeeded"
}
{
  "name": "nic-shortpat

## 3. Session-Host-Registrierung

**Erwartung:** `vmsp01` ist im Host Pool registriert, `Status = Available`, `allowNewSession = true`.


In [5]:
echo "=== Session Hosts in $HP ==="
# Diese az-CLI-Version hat kein 'sessionhost'-Kommando -> Management-API via az rest.
az rest --method GET \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.DesktopVirtualization/hostPools/${HP}/sessionHosts?api-version=2023-09-05" \
  --query "value[].{name:name,status:properties.status,allowNewSession:properties.allowNewSession,agentVersion:properties.agentVersion,osVersion:properties.osVersion}" -o json \
  | tee output/shortpath-sessionhosts.json


=== Session Hosts in hp-shortpath-cptdazavdvwan ===


[
  {
    "agentVersion": "1.0.14506.600",
    "allowNewSession": true,
    "name": "hp-shortpath-cptdazavdvwan/vmsp01",
    "osVersion": "10.0.26100.8457",
    "status": "Available"
  }
]


## 4. RBAC der AVD-Gruppe auf Shortpath-Ressourcen

**Erwartung:** `grp-avd-users-<prefix>` hat
- **Desktop Virtualization User** auf `dag-shortpath-`
- **Virtual Machine User Login** auf `vm-shortpath-`
- **Reader** auf der Resource Group (aus `main.bicep`)


In [6]:
GROUP_ID=$(az ad group list --filter "uniqueName eq 'grp-avd-users-${PREFIX}'" --query "[0].id" -o tsv)
echo "AVD-Gruppe grp-avd-users-${PREFIX}: ${GROUP_ID:-<nicht gefunden>}"
echo ""
echo "=== Rollen der Gruppe im RG-Scope (gefiltert auf shortpath) ==="
az role assignment list --assignee "$GROUP_ID" --all \
  --query "[?contains(scope,'shortpath') || ends_with(scope,'rg-${PREFIX}')].{role:roleDefinitionName,scope:scope}" -o json \
  | tee output/shortpath-rbac.json


AVD-Gruppe grp-avd-users-cptdazavdvwan: 5922ce8e-e370-40d2-95af-dd61243819c9

=== Rollen der Gruppe im RG-Scope (gefiltert auf shortpath) ===
[
  {
    "role": "Reader",
    "scope": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourcegroups/rg-cptdazavdvwan"
  },
  {
    "role": "Desktop Virtualization User",
    "scope": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourcegroups/rg-cptdazavdvwan/providers/Microsoft.DesktopVirtualization/applicationGroups/dag-shortpath-cptdazavdvwan"
  },
  {
    "role": "Virtual Machine User Login",
    "scope": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourcegroups/rg-cptdazavdvwan/providers/Microsoft.Compute/virtualMachines/vm-shortpath-cptdazavdvwan"
  }
]


## 5. Firewall-Relay-Regel (Public-Networks-Pfad)

RDP Shortpath für Public Networks benötigt ausgehend **UDP 3478 → 51.5.0.0/16** (TURN-Relay).

**Erwartung:** Die Netzwerkregel `rdp-shortpath-relay` in der Firewall-Policy erlaubt diesen Traffic aus dem AVD-Spoke (`10.1.0.0/16`), der auch den Shortpath-Host enthält.


In [7]:
echo "=== Firewall-Netzwerkregel 'rdp-shortpath-relay' ==="
az network firewall policy rule-collection-group collection list \
  --policy-name afwp-${PREFIX} -g $RG --rule-collection-group-name rcg-allow \
  --query "[].rules[?name=='rdp-shortpath-relay'].{name:name,proto:ipProtocols,src:sourceAddresses,dst:destinationAddresses,ports:destinationPorts}[]" -o json \
  2>/dev/null | tee output/shortpath-fw-rule.json

echo ""
echo "Hinweis: Quelle 10.1.0.0/16 (avdSpokeAddr) deckt den gesamten Spoke ab,"
echo "also auch nic-shortpath-${PREFIX} im Subnet snet-avd-hosts."


=== Firewall-Netzwerkregel 'rdp-shortpath-relay' ===
[
  {
    "dst": [
      "51.5.0.0/16"
    ],
    "name": "rdp-shortpath-relay",
    "ports": [
      "3478"
    ],
    "proto": [
      "UDP"
    ],
    "src": [
      "10.1.0.0/16"
    ]
  }
]

Hinweis: Quelle 10.1.0.0/16 (avdSpokeAddr) deckt den gesamten Spoke ab,
also auch nic-shortpath-cptdazavdvwan im Subnet snet-avd-hosts.


## 6. Effektive Routen + ausgehende IP

**Erwartung:** Der Shortpath-Host folgt demselben Routing wie der primäre Host (Routing Intent + NAT-Gateway-Bypass für WVD). Der TURN-Relay-Traffic (51.5.0.0/16) verlässt das Netz über die NAT-Gateway- bzw. Firewall-PIP.


In [8]:
echo "=== Effektive Routen auf $NIC ==="
az network nic show-effective-route-table -g $RG -n $NIC -o json \
  | tee output/shortpath-effective-routes.json \
  | jq -r '.value[] | select(.source != null) | "\(.addressPrefix[0])\t-> \(.nextHopType)\t(\(.source))"' 2>/dev/null \
  | head -20


=== Effektive Routen auf nic-shortpath-cptdazavdvwan ===
10.1.0.0/16	-> VnetLocal	(Default)
10.0.0.0/16	-> VNetPeering	(Default)
192.168.0.0/16	-> VirtualNetworkGateway	(VirtualNetworkGateway)
0.0.0.0/0	-> VirtualNetworkGateway	(VirtualNetworkGateway)
10.0.0.0/8	-> VirtualNetworkGateway	(VirtualNetworkGateway)
172.16.0.0/12	-> VirtualNetworkGateway	(VirtualNetworkGateway)
172.183.252.22/32	-> Internet	(User)
10.99.0.0/16	-> None	(User)


## 7. In-Guest: RDP-Transport-Default prüfen

Per `az vm run-command` wird auf dem Host der RDP-Transport-Status aus der Registry gelesen.

**Erwartung (Public Networks):**
- `UDP transport: DEFAULT (enabled)` → STUN/TURN nutzbar
- `Managed-networks listener: not set` → Managed-Pfad bewusst aus


In [9]:
echo "=== RDP-Transport-Default auf $VM (dauert ~30s) ==="
az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript --scripts "
  \$p='HKLM:\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services'
  \$v=Get-ItemProperty -Path \$p -Name fClientDisableUDP,fUseUdpPortRedirector,UdpPortNumber -ErrorAction SilentlyContinue
  if (\$null -eq \$v -or \$null -eq \$v.fClientDisableUDP) {
    Write-Output 'UDP transport: DEFAULT (enabled). Public-networks Shortpath (STUN/TURN) can be used.'
  } elseif (\$v.fClientDisableUDP -eq 1) {
    Write-Output 'UDP transport: DISABLED by policy (fClientDisableUDP=1). Falls back to TCP.'
  } else {
    Write-Output 'UDP transport: enabled (fClientDisableUDP=0).'
  }
  Write-Output ('Managed-networks listener (fUseUdpPortRedirector): ' + \$(if (\$v.fUseUdpPortRedirector) { \$v.fUseUdpPortRedirector } else { 'not set (managed-networks Shortpath off)' }))
" --query "value[].message" -o tsv | tee output/shortpath-transport.txt


=== RDP-Transport-Default auf vm-shortpath-cptdazavdvwan (dauert ~30s) ===
UDP transport: DEFAULT (enabled). Public-networks Shortpath (STUN/TURN) can be used.
Managed-networks listener (fUseUdpPortRedirector): not set (managed-networks Shortpath off)



## 8. Post-Connection-Verifikation (optional)

Erst **nach** einer aktiven Verbindung aussagekräftig: Event ID **135** im Log `Microsoft-Windows-RemoteDesktopServices-RdpCoreCDV/Operational` meldet `transport type set to UDP`, wenn Shortpath genutzt wurde.

> Ohne vorherige Sitzung ist das Log leer – das ist kein Fehler. Die endgültige Bestätigung erfolgt sonst client-seitig im *Connection Information*-Dialog (`UDP` / `UDP (Relay)` / `UDP (Private Network)`).


In [10]:
echo "=== Event ID 135 (RdpCoreCDV) auf $VM ==="
az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript --scripts "
  try {
    Get-WinEvent -LogName 'Microsoft-Windows-RemoteDesktopServices-RdpCoreCDV/Operational' -MaxEvents 200 -ErrorAction Stop |
      Where-Object Id -eq 135 |
      Select-Object TimeCreated, Message -First 5 | Format-List
  } catch { Write-Output 'Noch keine Shortpath-Verbindungen protokolliert (Log leer oder nicht vorhanden).' }
" --query "value[].message" -o tsv | tee output/shortpath-event135.txt


=== Event ID 135 (RdpCoreCDV) auf vm-shortpath-cptdazavdvwan ===
 \ Finished ..



## 9. Vergleich: Standard-Host vs. dedizierter Shortpath-Host

Hier werden **beide** Session-Hosts nebeneinander geprüft, um zu zeigen, ob RDP Shortpath ein- oder ausgeschaltet ist:

| Host | Pool | Zweck |
|------|------|-------|
| `vm-avd-` (vmavd01) | `hp-` | Produktiver Standard-Host |
| `vm-shortpath-` (vmsp01) | `hp-shortpath-` | Dedizierter Shortpath-Testhost |

Pro Host werden drei Dimensionen ausgewertet:

1. **Voraussetzung (UDP-Transport):** Registry `fClientDisableUDP`. Nicht gesetzt / `0` = UDP aktiv → Public-Networks-Shortpath (STUN/TURN) möglich. `1` = nur TCP.
2. **Managed-Networks-Listener:** Registry `fUseUdpPortRedirector` (UDP 3390). In diesem Lab bewusst **aus** auf beiden Hosts.
3. **Tatsächlich genutzt:** Event ID **135** im Log `RdpCoreCDV/Operational` belegt eine echte UDP-Sitzung (erst nach einem Login aussagekräftig).

> Beide Hosts liegen im selben Spoke `10.1.0.0/16` und werden von derselben Firewall-Relay-Regel (`rdp-shortpath-relay`, UDP 3478 → 51.5.0.0/16) abgedeckt. Der Unterschied ist also **nicht** die Netzwerkvorbereitung, sondern allein der dedizierte Host Pool/Workspace.
>
> Die endgültige Client-seitige Bestätigung erfolgt im AVD-Client unter *Connection Information* (`UDP` / `UDP (Relay)` / `UDP (Private Network)`).


In [12]:
echo "=== Shortpath-Status im Vergleich (dauert ~1 Min für beide Hosts) ==="
echo ""
: > output/shortpath-compare.txt

for PAIR in "Standard-Host|$VM_BASE" "Shortpath-Host|$VM"; do
  LABEL="${PAIR%%|*}"; H="${PAIR##*|}"
  echo "######## $LABEL : $H ########" | tee -a output/shortpath-compare.txt
  az vm run-command invoke -g "$RG" -n "$H" --command-id RunPowerShellScript --scripts "
    \$p='HKLM:\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services'
    \$v=Get-ItemProperty -Path \$p -Name fClientDisableUDP,fUseUdpPortRedirector,UdpPortNumber -ErrorAction SilentlyContinue
    if (\$null -eq \$v -or \$null -eq \$v.fClientDisableUDP) {
      Write-Output '1) Voraussetzung UDP-Transport : AKTIV (Default, fClientDisableUDP nicht gesetzt)'
    } elseif (\$v.fClientDisableUDP -eq 1) {
      Write-Output '1) Voraussetzung UDP-Transport : AUS (fClientDisableUDP=1 -> nur TCP)'
    } else {
      Write-Output '1) Voraussetzung UDP-Transport : AKTIV (fClientDisableUDP=0)'
    }
    Write-Output ('2) Managed-Networks-Listener   : ' + \$(if (\$v.fUseUdpPortRedirector) { 'AN (fUseUdpPortRedirector=' + \$v.fUseUdpPortRedirector + ')' } else { 'aus' }))
    try {
      \$e = Get-WinEvent -LogName 'Microsoft-Windows-RemoteDesktopServices-RdpCoreCDV/Operational' -MaxEvents 200 -ErrorAction Stop | Where-Object Id -eq 135 | Select-Object -First 1
      if (\$e) { Write-Output ('3) Tatsaechlich genutzt        : JA (Event 135 zuletzt ' + \$e.TimeCreated + ')') }
      else { Write-Output '3) Tatsaechlich genutzt        : noch keine UDP-Sitzung (kein Event 135)' }
    } catch { Write-Output '3) Tatsaechlich genutzt        : Log leer/nicht vorhanden (noch keine Verbindung)' }
  " --query "value[].message" -o tsv | tee -a output/shortpath-compare.txt
  echo "" | tee -a output/shortpath-compare.txt
done


=== Shortpath-Status im Vergleich (dauert ~1 Min für beide Hosts) ===

######## Standard-Host : vm-avd-cptdazavdvwan ########
1) Voraussetzung UDP-Transport : AKTIV (Default, fClientDisableUDP nicht gesetzt)
2) Managed-Networks-Listener   : aus
3) Tatsaechlich genutzt        : JA (Event 135 zuletzt 06/10/2026 15:11:42)


######## Shortpath-Host : vm-shortpath-cptdazavdvwan ########
1) Voraussetzung UDP-Transport : AKTIV (Default, fClientDisableUDP nicht gesetzt)
2) Managed-Networks-Listener   : aus
3) Tatsaechlich genutzt        : noch keine UDP-Sitzung (kein Event 135)




## 10. RDP Shortpath aktivieren / deaktivieren (In-Guest)

Public-Networks-Shortpath hängt am UDP-Transport. Über die Policy-Registry `fClientDisableUDP` lässt sich Shortpath gezielt **abschalten** oder wieder **einschalten** – ohne Neudeployment.

| Wert | Bedeutung |
|------|-----------|
| nicht gesetzt / `0` | UDP aktiv → Shortpath (STUN/TURN) möglich (**Default**) |
| `1` | UDP aus → RDP fällt auf **TCP** zurück, Shortpath deaktiviert |

`fClientDisableUDP` unter `HKLM\SOFTWARE\Policies\Microsoft\Windows NT\Terminal Services` ist die Registry-Entsprechung der GPO/Intune-Einstellung **„Select RDP transport protocols" → „Use only TCP"** (`ADMX_TerminalServer/TS_SELECT_TRANSPORT`). Laut Doku schalten diese Einstellungen genau diese Registry-Keys auf dem Session Host.

**Steuerung:** unten `ACTION` und `TARGET` setzen, dann Zelle ausführen.

- `ACTION=disable` → setzt `fClientDisableUDP=1`
- `ACTION=enable` → entfernt den Wert (zurück auf Default = UDP aktiv)
- `TARGET=$VM` (Shortpath-Host) oder `TARGET=$VM_BASE` (Standard-Host)

> Greift bei **neuen** Verbindungen; eine laufende Sitzung bleibt unverändert. Reine In-Guest-Registry-Änderung – kein Reboot nötig. Zum endgültigen Verifizieren danach Sektion 9 erneut ausführen.

### Anleitung zum Deaktivieren (offizielle Quellen)

Microsoft dokumentiert drei Wege, RDP Shortpath abzuschalten:

1. **Host-Pool-Ebene (empfohlen, kein In-Guest)** – per `Update-AzWvdHostPool` die UDP-Optionen auf `Disabled` setzen (`PublicUdp`/`RelayUdp` für Public Networks). Saubere, zentral steuerbare Variante: [Configure host pool networking settings](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath#configure-host-pool-networking-settings)
2. **Session-Host per GPO/Intune** – RDP-Shortpath-Typen auf **Disabled** setzen: [Configure RDP Shortpath via Intune/Group Policy](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath#configure-rdp-shortpath-for-public-networks-using-microsoft-intune-and-group-policy)
3. **UDP-Transport komplett aus** – „Select RDP transport protocols" → **Use only TCP**. Auf dem **Session Host** ist das der Registry-Wert `SelectTransport=1` (siehe [Check that UDP is enabled on session hosts](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath#check-that-udp-is-enabled-on-session-hosts)); auf dem **Client** der Wert `fClientDisableUDP=1` unter `...\Terminal Services\Client` (siehe [Check that UDP is enabled on Windows client devices](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath#check-that-udp-is-enabled-on-windows-client-devices)).

> **Hinweis:** Die Toggle-Zelle unten nutzt `fClientDisableUDP` als schnellen In-Guest-Schalter. Der **offiziell dokumentierte Session-Host-Schalter** ist jedoch `SelectTransport` (1 = „Use only TCP"). Für eine zentrale, IaC-freundliche Deaktivierung ist Weg 1 (`Update-AzWvdHostPool`) am saubersten.

**Allgemeine Referenzen:**

- [Configure RDP Shortpath – inkl. „verify it is working" und „disable it if needed"](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath)
- [RDP Shortpath – Funktionsweise (STUN/TURN, public vs. managed networks)](https://learn.microsoft.com/azure/virtual-desktop/rdp-shortpath)
- [Policy CSP – ADMX_TerminalServer/TS_SELECT_TRANSPORT (UDP komplett abschalten → „Use only TCP")](https://learn.microsoft.com/windows/client-management/mdm/policy-csp-admx-terminalserver#ts_select_transport)
- [Use the administrative template for Azure Virtual Desktop (GPO/Intune Setup)](https://learn.microsoft.com/azure/virtual-desktop/administrative-template)


In [ ]:
# ----- Steuerung -----
ACTION=disable        # disable | enable
TARGET=$VM            # $VM = Shortpath-Host (vmsp01) | $VM_BASE = Standard-Host (vmavd01)
# ---------------------

echo "=== Shortpath '$ACTION' auf $TARGET (dauert ~30s) ==="
if [ "$ACTION" = "disable" ]; then
  PS="
    \$p='HKLM:\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services'
    New-Item -Path \$p -Force | Out-Null
    New-ItemProperty -Path \$p -Name fClientDisableUDP -PropertyType DWord -Value 1 -Force | Out-Null
    Write-Output 'RDP Shortpath DEAKTIVIERT: fClientDisableUDP=1 (UDP aus, nur TCP). Gilt fuer neue Verbindungen.'
  "
elif [ "$ACTION" = "enable" ]; then
  PS="
    \$p='HKLM:\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services'
    Remove-ItemProperty -Path \$p -Name fClientDisableUDP -ErrorAction SilentlyContinue
    Write-Output 'RDP Shortpath AKTIVIERT: fClientDisableUDP entfernt (UDP-Transport Default = an).'
  "
else
  echo "Unbekannte ACTION '$ACTION' (erlaubt: disable | enable)"; PS=""
fi

if [ -n "$PS" ]; then
  az vm run-command invoke -g "$RG" -n "$TARGET" --command-id RunPowerShellScript \
    --scripts "$PS" --query "value[].message" -o tsv | tee output/shortpath-toggle.txt
fi
